In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

S&lection des actifs

In [ ]:
print ("================= Welcome to the Portfolio Risk Engine!=====================")
print ()
print ("Please enter the tickers of your portfolio assets (separated by commas):")
actif1 = input('Enter the first asset ticker: ')
actif2 = input('Enter the second asset ticker: ')
actif3 = input('Enter the third asset ticker: ')
actif4 = input('Enter the fourth asset ticker: ')
actif5 = input('Enter the fifth asset ticker: ')
actif6 = input('Enter the sixth asset ticker: ')

================= Welcome to the Portfolio Risk Engine!=====================

Please enter the tickers of your portfolio assets (separated by commas):


In [3]:
tickers = ('AAPL', 'JPM','JNJ','AMZN','PG','NEE','CAT','AMT','GLD')
data = yf.download (tickers, start='2020-01-01', end='2025-12-31')['Close']
data = pd.DataFrame(data).dropna()
data

[*********************100%***********************]  9 of 9 completed


Ticker,AAPL,AMT,AMZN,CAT,GLD,JNJ,JPM,NEE,PG
Date,,,,,,,,,
2020-01-02,72.271530,187.878433,94.900497,132.067856,143.949997,121.338516,117.313942,50.335991,103.959297
2020-01-03,71.568901,187.968826,93.748497,130.234207,145.860001,119.933685,115.765816,50.694592,103.260109
2020-01-06,72.139198,187.919510,95.143997,130.146454,147.389999,119.784073,115.673729,50.947735,103.403320
2020-01-07,71.799927,183.915314,95.343002,128.426849,147.970001,120.515564,113.707260,50.903439,102.763115
2020-01-08,72.954918,185.510406,94.598503,129.567413,146.860001,120.498978,114.594276,50.880222,103.201141
...,...,...,...,...,...,...,...,...,...
2025-12-23,271.620667,167.687576,232.139999,578.882446,413.640015,202.522537,319.771271,78.086189,140.062469
2025-12-24,273.066711,167.966232,232.380005,580.214355,411.929993,204.490891,322.950104,78.732086,141.343933
2025-12-26,272.657867,169.513290,232.520004,579.458984,416.739990,204.343277,321.713867,78.692955,141.588516


Poids des actifs

In [ ]:
w1 = float(input(f'Enter the weight of {actif1}: '))
w2 = float(input(f'Enter the weight of {actif2}: '))
w3 = float(input(f'Enter the weight of {actif3}: '))
w4 = float(input(f'Enter the weight of {actif4}: '))
w5 = float(input(f'Enter the weight of {actif5}: '))
w6 = float(input(f'Enter the weight of {actif6}: '))

if w1 + w2 + w3 + w4 + w5 + w6 != 1:
    print("Error: The sum of the weights must be equal to 1.")

Calcul du rendement du portefeuille

In [9]:
assets_returns = data.pct_change().dropna()
assets_returns = assets_returns.mean (axis=0)
print (assets_returns)


Ticker
AAPL    0.001081
AMT     0.000114
AMZN    0.000848
CAT     0.001184
GLD     0.000730
JNJ     0.000420
JPM     0.000856
NEE     0.000477
PG      0.000286
dtype: float64


In [19]:
portfolio_weights = np.array([0.15, 0.1, 0.2, 0.1, 0.2, 0.1, 0.05, 0.05, 0.05])  # Example weights for the assets
portfolio = pd.DataFrame ({'Assets': data.columns, 'Returns': assets_returns, 'Weights': portfolio_weights})

In [23]:
portfolio_returns = portfolio['Returns']*portfolio['Weights']
print(f"Expected Portfolio Return: {portfolio_returns.sum():.4f}")

Expected Portfolio Return: 0.0007


Monte Carlo VaR

In [ ]:
def simulation_gbm (weights, returns, cov, S0 = 1.0, n_days = 252, n_sim = 10000) : 
    np.random.seed (42)
    dt = 1/252
    
    port_values = np.zeros ((n_sim, n_days+1))
    port_values [:,0] = S0    #n_sim simulations de la valeur de l'actif sur n_days

    for t in range (1, n_days + 1) :
        asset_returns = (returns - 0.5 * np.diag(cov)) * dt + np.diag(cov) * np.sqrt (dt)
        port_return = asset_returns @ weights     
        port_values [:,t] = port_values [:, t-1] * np.exp (port_return)

    return port_values

In [76]:
simulated_portfolio_values = simulation_gbm (portfolio_weights, assets_returns, data.cov())

C:\Users\merry\AppData\Local\Temp\ipykernel_70024\739139403.py:15: RuntimeWarning: overflow encountered in multiply
  port_values [:,t] = port_values [:, t-1] * np.exp (port_return)


In [114]:
port_values = np.zeros ((10, 90))
port_values [:,0] = 1.0    #n_sim simulations de la valeur de l'actif sur n_days

#print (port_values)
asset_ret = (assets_returns - 0.5 * np.diag(data.cov())) * 1/252 + np.diag(data.cov()) * np.sqrt (1/252)
port_return = asset_ret @ portfolio_weights
#port_values [:,2] = port_values [:, 1] * np.exp (port_return)

for t in range (1,90) : 
    port_values [:,t] = port_values [:, t-1] * np.exp (port_return)

print (t)
print (port_values)

89
[[1.00000000e+000 1.33819450e+074 1.79076453e+148 2.39639125e+222
  3.20683760e+296             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
              inf             inf             inf             inf
       

C:\Users\merry\AppData\Local\Temp\ipykernel_70024\3831308945.py:10: RuntimeWarning: overflow encountered in multiply
  port_values [:,t] = port_values [:, t-1] * np.exp (port_return)


In [93]:
print (port_return)

170.6826182013818


In [81]:
print (simulated_portfolio_values [-1])

[1.00000000e+000 1.33819450e+074 1.79076453e+148 2.39639125e+222
 3.20683760e+296             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf             inf             inf             inf
             inf         

In [ ]:
ret = np.array(assets_returns.values)

asset_ret = (ret - 0.5 * np.diag(data.cov())) * (1/252) + np.diag(data.cov()) * np.sqrt (1/252)
asset_ret = (assets_returns - 0.5 * np.diag(data.cov()))
print (asset_ret)


Ticker
AAPL   -1226.309139
AMT     -254.510223
AMZN    -798.536714
CAT    -5706.534145
GLD    -1689.337245
JNJ     -125.137677
JPM    -1971.597216
NEE      -35.009557
PG      -166.783701
dtype: float64
